In [1]:
import gpboost as gpb
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
# Load data
data = pd.read_csv('C:/Files/GLAN_DATA/first_wave/downscale_data_60/data_with_activity_sequence_all_recod_downsampling_activity_travel.csv')
data = data[data['time_weight'] >= 5]
data = data[data['tree_height'] >= 0]
print(data.shape)
pred_vars = ['household_income', 'age', 'gender', 'education_level', 'employment_status', 'neighborhood_type',
             'work_study', 'housework', 'personal_affair', 'leisure',
             'travel', 'transportation', 'residence', 'industry', 'company', 'shopping',
             'restaurant', 'life_service', 'education_culture', 'entertainment', 'sport_fitness',
             'recreation_tourism', 'healthcare', 'workday', 'time_hour', 'mobility_status',
             'time_nonflexibility', 'home_ornot', 'POI_density', 'POI_diversity', "tree_height"]
# Prepare 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=666)
fold_results = []


(8427, 52)


C:\anaconda3\envs\torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import gpboost as gpb
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import pickle
import warnings

warnings.filterwarnings('ignore')

# Best parameters
best_params = {
    'num_boost_round': 280,
    'learning_rate': 0.001292112722051777,
    'max_depth': 5,
    'num_leaves': 127,
    'min_data_in_leaf': 45,
    'lambda_l1': 22.605341824915648,
    'lambda_l2': 7.170332586173781,
    'feature_fraction': 0.7386048051720877,
    'min_gain_to_split': 4.374371803180026,
    'min_sum_hessian_in_leaf': 1.7580285746703073
}


final_params = best_params.copy()
final_params.update({'verbose': -1, 'objective': 'regression', 'metric': 'mse'})

# Bootstrap settings
n_bootstrap = 1000
models_dir = 'models'
os.makedirs(models_dir, exist_ok=True)


gp_model_orig = gpb.GPModel(group_data=data['pid'], likelihood='gaussian')
data_bst_orig = gpb.Dataset(data=data[pred_vars], label=data['compound_exposure_disadvantage'])
gpbst_orig = gpb.train(
    params=final_params,
    train_set=data_bst_orig,
    gp_model=gp_model_orig
)


# Predict on the full data (fixed + random effects)
y_hat = gpbst_orig.predict(data=data[pred_vars], group_data_pred=data['pid'], pred_latent=False)['response_mean']
residuals = data['compound_exposure_disadvantage'] - y_hat


for i in tqdm(range(n_bootstrap), desc="Training models"):
    
    rng = np.random.default_rng(seed=42 + i)
    
    
    res_boot = rng.choice(residuals, size=len(residuals), replace=True)
    
    
    y_boot = y_hat + res_boot
    
    
    gp_model = gpb.GPModel(group_data=data['pid'], likelihood='gaussian')
    data_bst = gpb.Dataset(data=data[pred_vars], label=y_boot)
    
    
    gpbst = gpb.train(
        params=final_params,
        train_set=data_bst,
        gp_model=gp_model
    )
    
    # Save model
    path = os.path.join(models_dir, f"gpb_model_{i}.sav")
    with open(path, 'wb') as f:
        pickle.dump(gpbst, f)

print("All models trained and saved.")

Training models: 100%|██████████| 1000/1000 [06:35<00:00,  2.53it/s]

All models trained and saved.
